# 06 — RL Evaluation, Trade Analysis & Strategy Comparison (AAPL)

This notebook:
1. Loads RL agent results (DQN + PPO) from `05_rl_trading_agent.ipynb`
2. Re-runs all 4 rule-based strategies (S1–S4) from `04_rule_based_strategies_optimized.ipynb` on the **same test period** for fair comparison
3. Plots successful trades and returns for each RL agent
4. Generates comprehensive comparison plots and tables:
   - Equity curves
   - Drawdown profiles
   - Monthly return heatmaps
   - Risk-Return scatter
   - Trade statistics

---
## 🎓 Assignment Instructions

This notebook has **6 TODOs**, marked with `# TODO N (DIFFICULTY):` comments
and `raise NotImplementedError(...)`. Everything else (loading results,
re-running the rule-based strategies from notebook 04, and all the plotting
scaffolding around each TODO) is already implemented — you only need to fill
in the specific lines described in the markdown cell directly above each
TODO.

**Difficulty breakdown:** 4 × EASY, 2 × MEDIUM.

| # | Difficulty | Location | What you're implementing |
|---|---|---|---|
| 8 | EASY | Summary comparison table | Build `summary_df` from all strategies' metrics |
| 9 | EASY | `extract_trades()` | Per-trade PnL and % return |
| 10 | MEDIUM | Equity curves plot | Normalize portfolio values to cumulative % return |
| 11 | EASY | Drawdown profiles plot | Running peak and drawdown % |
| 12 | EASY | Risk-return scatter plot | Annualized volatility |
| 13 | MEDIUM | Rolling Sharpe plot | Rolling-window Sharpe ratio |

**Prerequisite:** this notebook loads `reports/rl_results.pkl`, produced by
running notebook `05` (with its own TODOs completed) end-to-end at least
once. Run `05` first if that file doesn't exist yet.

**How to know you're done:** a cell with an unfinished TODO will raise
`NotImplementedError` when run. Once every TODO is filled in, all cells
should run top-to-bottom without errors and produce the comparison table
and plots.
---

In [ ]:
# ─── IMPORTS ─────────────────────────────────────────────────────────────────
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# ── Plotting style ────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor':   '#161b22',
    'axes.edgecolor':   '#30363d',
    'axes.labelcolor':  '#e6edf3',
    'axes.titlecolor':  '#e6edf3',
    'xtick.color':      '#8b949e',
    'ytick.color':      '#8b949e',
    'grid.color':       '#21262d',
    'grid.alpha':       1.0,
    'text.color':       '#e6edf3',
    'legend.facecolor': '#21262d',
    'legend.edgecolor': '#30363d',
    'legend.labelcolor':'#e6edf3',
    'font.family':      'DejaVu Sans',
    'axes.grid':        True,
    'axes.titlesize':   12,
    'axes.labelsize':   10,
})

# Color palette
COLORS = {
    'DQN':            '#00d2ff',   # cyan
    'PPO':            '#a78bfa',   # purple
    'S1':             '#fb923c',   # orange
    'S2':             '#34d399',   # green
    'S3':             '#f472b6',   # pink
    'S4':             '#fbbf24',   # yellow
    'Buy&Hold':       '#64748b',   # gray
    'profit':         '#22c55e',   # bright green
    'loss':           '#ef4444',   # bright red
}

BASE_DIR    = os.path.abspath('.')
DATA_DIR    = os.path.join(BASE_DIR, 'data', 'processed')
MODEL_DIR   = os.path.join(BASE_DIR, 'models')
REPORTS_DIR = os.path.join(BASE_DIR, 'reports')

print('Environment ready.')

## 1  Load All Results

In [ ]:
# ─── LOAD RL RESULTS ─────────────────────────────────────────────────────────
with open(os.path.join(REPORTS_DIR, 'rl_results.pkl'), 'rb') as f:
    rl_results = pickle.load(f)

dqn_res = rl_results['DQN']
ppo_res = rl_results['PPO']
test_dates  = pd.to_datetime(rl_results['test_dates'])
test_prices = rl_results['test_prices']
test_start_idx = rl_results['test_start_idx']
test_end_idx   = rl_results['test_end_idx']

print(f'Test period: {test_dates[0].date()} → {test_dates[-1].date()} ({len(test_dates)} days)')
print()
print(f'DQN  — Return: {dqn_res["total_return"]:+.2f}%  Sharpe: {dqn_res["sharpe"]:.3f}  MDD: {dqn_res["max_drawdown"]:.2f}%')
print(f'PPO  — Return: {ppo_res["total_return"]:+.2f}%  Sharpe: {ppo_res["sharpe"]:.3f}  MDD: {ppo_res["max_drawdown"]:.2f}%')

In [ ]:
# ─── LOAD AAPL DATA ──────────────────────────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'AAPL_features.csv'), parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

N       = len(df)
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
n_train = int(N * TRAIN_RATIO)
n_val   = int(N * VAL_RATIO)

test_df = df.iloc[n_train + n_val:].reset_index(drop=True)
print(f'Test set: {len(test_df)} rows')

In [ ]:
# ─── RULE-BASED STRATEGIES (re-implemented from 04) ──────────────────────────
# These match the strategies in 04_rule_based_strategies_optimized.ipynb

INITIAL_CAPITAL  = 100_000.0
TRANSACTION_COST = 0.001

def backtest_strategy(signals, prices, dates, name='Strategy'):
    """
    Generic vectorized backtester.
    signals: array of +1 (long), -1 (short/exit), 0 (hold)
    Returns full metrics dict.
    """
    capital  = INITIAL_CAPITAL
    position = 0.0      # shares held
    pv_hist  = [capital]
    trades   = []
    buy_price = None

    for i, (sig, px, dt) in enumerate(zip(signals, prices, dates)):
        if sig == 1 and position == 0:   # BUY
            invest   = capital
            position = invest * (1 - TRANSACTION_COST) / px
            capital  = 0.0
            buy_price = px
            trades.append({'date': dt, 'action': 'BUY', 'price': px})

        elif sig == -1 and position > 0:  # SELL
            proceeds = position * px * (1 - TRANSACTION_COST)
            capital  = proceeds
            pnl      = px - buy_price
            trades.append({'date': dt, 'action': 'SELL', 'price': px, 'pnl': pnl})
            position = 0.0
            buy_price = None

        pv = capital + position * px
        pv_hist.append(pv)

    pvh = np.array(pv_hist)
    dr  = np.diff(pvh) / pvh[:-1]

    total_return = (pvh[-1] / pvh[0] - 1) * 100
    ann_return   = ((pvh[-1] / pvh[0]) ** (252 / max(len(pvh), 1)) - 1) * 100
    sharpe       = (dr.mean() / (dr.std() + 1e-8)) * np.sqrt(252) if len(dr) > 0 else 0.0
    peak         = np.maximum.accumulate(pvh)
    dd           = (pvh - peak) / peak
    max_dd       = float(dd.min() * 100)

    sell_trades = [t for t in trades if t.get('action') == 'SELL']
    pnls        = [t.get('pnl', 0) for t in sell_trades]
    win_rate    = (np.array(pnls) > 0).mean() * 100 if pnls else 0.0

    return {
        'name':             name,
        'total_return':     total_return,
        'ann_return':       ann_return,
        'sharpe':           sharpe,
        'max_drawdown':     max_dd,
        'win_rate':         win_rate,
        'n_trades':         len(sell_trades),
        'portfolio_values': pvh,
        'daily_returns':    dr,
        'trade_history':    trades,
        'dates':            dates,
    }


# ── Fetch test data arrays ────────────────────────────────────────────────────
px   = test_df['close'].values
dt   = test_df['date'].values
rsi  = test_df['rsi_14'].values
macd = test_df['macd'].values
macd_sig = test_df['macd_signal'].values
bb_pct   = test_df['bb_pct_b'].values
ret5     = test_df['return_5'].values
atr      = test_df['atr_14'].values
sma20    = test_df['sma_20'].values
sma50    = test_df['sma_50'].values
sma200   = test_df['sma_200'].values
bb_upper = test_df['bb_upper'].values
bb_lower = test_df['bb_lower'].values
atr_r    = test_df['atr_ratio'].values

print('Test data arrays ready.')

In [ ]:
# ─── S1: REGIME-ROUTED LSTM SIGNAL ───────────────────────────────────────────
# Signal: buy if SMA20 > SMA50 (golden cross zone) and RSI not overbought
def s1_signals(test_df):
    sig = np.zeros(len(test_df), dtype=int)
    pos = False
    for i in range(1, len(test_df)):
        row = test_df.iloc[i]
        above_sma = row['sma_20'] > row['sma_50']
        rsi_ok    = 40 < row['rsi_14'] < 70
        macd_pos  = row['macd'] > row['macd_signal']

        if not pos and above_sma and rsi_ok and macd_pos:
            sig[i] = 1;  pos = True
        elif pos and (not above_sma or row['rsi_14'] > 75):
            sig[i] = -1; pos = False
    return sig

s1_sig = s1_signals(test_df)
s1_res = backtest_strategy(s1_sig, px, dt, name='S1: Regime Routed')
print(f'S1 Return: {s1_res["total_return"]:+.2f}%  Sharpe: {s1_res["sharpe"]:.3f}')

In [ ]:
# ─── S2: MOMENTUM + LSTM SIGNAL ──────────────────────────────────────────────
def s2_signals(test_df):
    sig = np.zeros(len(test_df), dtype=int)
    pos = False
    for i in range(20, len(test_df)):
        row   = test_df.iloc[i]
        prev  = test_df.iloc[i-1]
        macd_cross_up   = (row['macd'] > row['macd_signal']) and (prev['macd'] <= prev['macd_signal'])
        macd_cross_down = (row['macd'] < row['macd_signal']) and (prev['macd'] >= prev['macd_signal'])
        strong_up = row['return_5'] > 0.01 and row['rsi_14'] > 50
        strong_dn = row['return_5'] < -0.01 and row['rsi_14'] < 50

        if not pos and (macd_cross_up or strong_up):
            sig[i] = 1;  pos = True
        elif pos and (macd_cross_down or strong_dn or row['rsi_14'] > 78):
            sig[i] = -1; pos = False
    return sig

s2_sig = s2_signals(test_df)
s2_res = backtest_strategy(s2_sig, px, dt, name='S2: Momentum+LSTM')
print(f'S2 Return: {s2_res["total_return"]:+.2f}%  Sharpe: {s2_res["sharpe"]:.3f}')

In [ ]:
# ─── S3: MEAN REVERSION ───────────────────────────────────────────────────────
def s3_signals(test_df):
    sig = np.zeros(len(test_df), dtype=int)
    pos = False
    for i in range(1, len(test_df)):
        row = test_df.iloc[i]
        oversold   = row['rsi_14'] < 30 and row['bb_pct_b'] < 0.1
        overbought = row['rsi_14'] > 70 or row['bb_pct_b'] > 0.9

        if not pos and oversold:
            sig[i] = 1;  pos = True
        elif pos and overbought:
            sig[i] = -1; pos = False
    return sig

s3_sig = s3_signals(test_df)
s3_res = backtest_strategy(s3_sig, px, dt, name='S3: Mean Reversion')
print(f'S3 Return: {s3_res["total_return"]:+.2f}%  Sharpe: {s3_res["sharpe"]:.3f}')

In [ ]:
# ─── S4: VOLATILITY BREAKOUT ──────────────────────────────────────────────────
def s4_signals(test_df):
    sig = np.zeros(len(test_df), dtype=int)
    pos = False
    for i in range(20, len(test_df)):
        row  = test_df.iloc[i]
        prev = test_df.iloc[i-1]
        high_vol = row['atr_ratio'] > 0.015
        breakout_up = row['close'] > prev['bb_upper'] and high_vol and row['macd'] > 0
        breakout_dn = row['close'] < prev['bb_lower'] or row['atr_ratio'] < 0.008

        if not pos and breakout_up:
            sig[i] = 1;  pos = True
        elif pos and breakout_dn:
            sig[i] = -1; pos = False
    return sig

s4_sig = s4_signals(test_df)
s4_res = backtest_strategy(s4_sig, px, dt, name='S4: Vol Breakout')
print(f'S4 Return: {s4_res["total_return"]:+.2f}%  Sharpe: {s4_res["sharpe"]:.3f}')

In [ ]:
# ─── BUY & HOLD BASELINE ─────────────────────────────────────────────────────
bh_pvh = INITIAL_CAPITAL * (px / px[0])
bh_dr  = np.diff(bh_pvh) / bh_pvh[:-1]
bh_res = {
    'name':             'Buy & Hold',
    'total_return':     (bh_pvh[-1] / bh_pvh[0] - 1) * 100,
    'ann_return':       ((bh_pvh[-1] / bh_pvh[0]) ** (252 / len(bh_pvh)) - 1) * 100,
    'sharpe':           (bh_dr.mean() / (bh_dr.std() + 1e-8)) * np.sqrt(252),
    'max_drawdown':     float(((bh_pvh - np.maximum.accumulate(bh_pvh)) / np.maximum.accumulate(bh_pvh)).min() * 100),
    'win_rate':         (bh_dr > 0).mean() * 100,
    'n_trades':         1,
    'portfolio_values': np.concatenate([[INITIAL_CAPITAL], bh_pvh]),
    'daily_returns':    bh_dr,
    'trade_history':    [{'date': dt[0], 'action': 'BUY', 'price': px[0]}],
    'dates':            dt,
}
print(f'B&H Return: {bh_res["total_return"]:+.2f}%  Sharpe: {bh_res["sharpe"]:.3f}')

## 2  Summary Comparison Table

> ### 📝 TODO 8 (EASY) — Build the comparison DataFrame
>
> Build `summary_df` as a `pd.DataFrame` with one row per strategy (using the
> `labels` list already defined above) and these columns, each pulled out of
> `all_results` with a list comprehension over the metric's dictionary key:
> `'Strategy'` (= `labels`), `'Total Return (%)'` (key `'total_return'`),
> `'Ann. Return (%)'` (key `'ann_return'`), `'Sharpe Ratio'` (key `'sharpe'`),
> `'Max Drawdown (%)'` (key `'max_drawdown'`), `'Win Rate (%)'`
> (key `'win_rate'`), and `'# Trades'` (key `'n_trades'`). Every `r` in
> `all_results` is a metrics dict with exactly these keys (see `run_episode()`
> in notebook 05 if you want to double check).

In [ ]:
# ─── BUILD COMPARISON DATAFRAME ──────────────────────────────────────────────
all_results = [dqn_res, ppo_res, s1_res, s2_res, s3_res, s4_res, bh_res]
labels      = ['DQN (RL)', 'PPO (RL)', 'S1: Regime', 'S2: Momentum', 'S3: MeanRev', 'S4: VolBreak', 'Buy & Hold']

# TODO 8 (EASY): build summary_df — see the markdown cell above for the columns.
raise NotImplementedError("TODO 8: build the summary_df comparison DataFrame")

summary_df = summary_df.sort_values('Sharpe Ratio', ascending=False).reset_index(drop=True)
summary_df.to_csv(os.path.join(REPORTS_DIR, 'full_strategy_comparison.csv'), index=False)

# Display with styling
print('=== Strategy Comparison (Test Period) ===')
print(summary_df.to_string(index=False, float_format=lambda x: f'{x:>8.2f}'))

## 3  Plot: RL Successful Trades

> ### 📝 TODO 9 (EASY) — Trade PnL and percentage return
>
> Given a completed buy-sell pair with prices `buy_px` and `sell_px`,
> compute:
> - `pnl` — the absolute profit/loss **per share**: `sell_px - buy_px`.
> - `pct` — the percentage return: `(sell_px / buy_px - 1) * 100`.

In [ ]:
# ─── HELPER: PARSE TRADE HISTORY ─────────────────────────────────────────────
def extract_trades(trade_history, all_dates, all_prices):
    """
    Extract completed buy-sell pairs with profit/loss classification.
    Returns list of dicts with: buy_date, sell_date, buy_price, sell_price, pnl, pct_return
    """
    completed = []
    pending_buy = None
    for t in trade_history:
        if t['action'] == 'BUY':
            pending_buy = t
        elif t['action'] == 'SELL' and pending_buy is not None:
            buy_px  = pending_buy['price']
            sell_px = t['price']
            # TODO 9 (EASY): compute pnl (absolute $ profit/loss per share) and
            # pct (percentage return) — see the markdown cell above.
            raise NotImplementedError("TODO 9: compute pnl and pct in extract_trades()")
            completed.append({
                'buy_date':   pd.to_datetime(pending_buy['date']),
                'sell_date':  pd.to_datetime(t['date']),
                'buy_price':  buy_px,
                'sell_price': sell_px,
                'pnl':        pnl,
                'pct_return': pct,
                'profitable': pnl > 0
            })
            pending_buy = None
    return completed

all_dates_ts = pd.to_datetime(test_df['date'].values)
dqn_trades   = extract_trades(dqn_res['trade_history'], all_dates_ts, px)
ppo_trades   = extract_trades(ppo_res['trade_history'], all_dates_ts, px)

print(f'DQN completed trades: {len(dqn_trades)} | Win: {sum(1 for t in dqn_trades if t["profitable"])} | Loss: {sum(1 for t in dqn_trades if not t["profitable"])}')
print(f'PPO completed trades: {len(ppo_trades)} | Win: {sum(1 for t in ppo_trades if t["profitable"])} | Loss: {sum(1 for t in ppo_trades if not t["profitable"])}')

In [ ]:
# ─── PLOT: DQN TRADE VISUALIZATION ───────────────────────────────────────────
def plot_trades(agent_name, trades, test_df, portfolio_values, color, save_path):
    dates = pd.to_datetime(test_df['date'].values)
    prices = test_df['close'].values

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(3, 1, height_ratios=[3, 1.5, 1], hspace=0.08)

    # ─ Top: Price + trade markers ────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.plot(dates, prices, color='#475569', linewidth=1.0, alpha=0.8, label='AAPL Price')
    ax1.fill_between(dates, prices, prices.min(), alpha=0.05, color=color)

    profit_buys  = [t for t in trades if t['profitable']]
    profit_sells = [t for t in trades if t['profitable']]
    loss_buys    = [t for t in trades if not t['profitable']]
    loss_sells   = [t for t in trades if not t['profitable']]

    for t in trades:
        c = COLORS['profit'] if t['profitable'] else COLORS['loss']
        # Vertical span for holding period
        ax1.axvspan(t['buy_date'], t['sell_date'], alpha=0.08, color=c)
        # Buy marker
        ax1.scatter(t['buy_date'],  t['buy_price'],  marker='^', s=120, color=COLORS['profit'], zorder=5)
        # Sell marker
        ax1.scatter(t['sell_date'], t['sell_price'], marker='v', s=120, color=c, zorder=5)
        # Annotation for large trades
        if abs(t['pct_return']) > 2:
            ax1.annotate(f"{t['pct_return']:+.1f}%",
                         xy=(t['sell_date'], t['sell_price']),
                         xytext=(0, 15), textcoords='offset points',
                         fontsize=7, color=c, ha='center',
                         arrowprops=dict(arrowstyle='->', color=c, lw=0.8))

    buy_patch  = mpatches.Patch(color=COLORS['profit'], label='Buy entry', alpha=0.8)
    win_patch  = mpatches.Patch(color=COLORS['profit'], alpha=0.4, label='Profitable trade')
    loss_patch = mpatches.Patch(color=COLORS['loss'],   alpha=0.4, label='Loss trade')
    ax1.legend(handles=[buy_patch, win_patch, loss_patch], loc='upper left', fontsize=9)

    n_win  = sum(1 for t in trades if t['profitable'])
    n_loss = len(trades) - n_win
    avg_ret = np.mean([t['pct_return'] for t in trades]) if trades else 0
    ax1.set_title(f'{agent_name} — Trade Visualization on AAPL (Test Period)\n'
                  f'Trades: {len(trades)} | Wins: {n_win} | Losses: {n_loss} | Avg Return/Trade: {avg_ret:+.2f}%',
                  fontsize=12, fontweight='bold', pad=10)
    ax1.set_ylabel('Price ($)')
    ax1.set_xticklabels([])
    ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))

    # ─ Middle: Portfolio value vs B&H ────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    pv_dates = dates[:len(portfolio_values)-1] if len(portfolio_values)-1 <= len(dates) else dates
    ax2.plot(pv_dates, portfolio_values[1:len(pv_dates)+1], color=color, linewidth=1.5, label=f'{agent_name} Portfolio')
    bh_pv = 100_000 * (prices / prices[0])
    ax2.plot(dates[:len(bh_pv)], bh_pv, color=COLORS['Buy&Hold'], linewidth=1.0, linestyle='--', alpha=0.7, label='Buy & Hold')
    ax2.axhline(100_000, color='white', linewidth=0.5, alpha=0.3)
    ax2.fill_between(pv_dates,
                     portfolio_values[1:len(pv_dates)+1],
                     100_000, alpha=0.1, color=color)
    ax2.legend(fontsize=9, loc='upper left')
    ax2.set_ylabel('Portfolio ($)')
    ax2.set_xticklabels([])
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'${x/1000:.0f}K'))

    # ─ Bottom: Per-trade P&L bar chart ────────────────────────────────────────
    ax3 = fig.add_subplot(gs[2])
    if trades:
        sell_dates = [t['sell_date'] for t in trades]
        pnl_pcts   = [t['pct_return'] for t in trades]
        bar_colors = [COLORS['profit'] if p > 0 else COLORS['loss'] for p in pnl_pcts]
        ax3.bar(sell_dates, pnl_pcts, color=bar_colors, alpha=0.85, width=5)
        ax3.axhline(0, color='white', linewidth=0.5)
    ax3.set_ylabel('Trade Return (%)')
    ax3.set_xlabel('Date')
    ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax3.xaxis.get_majorticklabels(), rotation=30, ha='right')

    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print(f'Saved: {os.path.basename(save_path)}')


plot_trades(
    'DQN Agent', dqn_trades, test_df,
    dqn_res['portfolio_values'], COLORS['DQN'],
    os.path.join(REPORTS_DIR, 'dqn_trades_analysis.png')
)

In [ ]:
# ─── PPO TRADE VISUALIZATION ─────────────────────────────────────────────────
plot_trades(
    'PPO Agent', ppo_trades, test_df,
    ppo_res['portfolio_values'], COLORS['PPO'],
    os.path.join(REPORTS_DIR, 'ppo_trades_analysis.png')
)

## 4  Plot: Trade Return Distributions

In [ ]:
# ─── TRADE RETURN DISTRIBUTIONS ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Trade Return Distributions — DQN vs PPO (AAPL Test Set)',
             fontsize=13, fontweight='bold', color='#e6edf3')

for ax, (agent, trades, color) in zip(axes, [
    ('DQN', dqn_trades, COLORS['DQN']),
    ('PPO', ppo_trades, COLORS['PPO'])
]):
    if trades:
        pcts = [t['pct_return'] for t in trades]
        wins  = [p for p in pcts if p > 0]
        losses= [p for p in pcts if p <= 0]

        ax.hist(losses, bins=20, color=COLORS['loss'], alpha=0.7, label=f'Losses ({len(losses)})')
        ax.hist(wins,   bins=20, color=COLORS['profit'], alpha=0.7, label=f'Wins ({len(wins)})')
        ax.axvline(0, color='white', linewidth=1.0)
        ax.axvline(np.mean(pcts), color=color, linewidth=1.5, linestyle='--',
                   label=f'Mean: {np.mean(pcts):+.2f}%')

        # Stats box
        wr   = len(wins) / len(pcts) * 100 if pcts else 0
        avg  = np.mean(pcts)
        best = max(pcts)
        worst= min(pcts)
        textstr = f'Trades: {len(pcts)}\nWin Rate: {wr:.1f}%\nAvg: {avg:+.2f}%\nBest: {best:+.2f}%\nWorst: {worst:+.2f}%'
        props = dict(boxstyle='round,pad=0.5', facecolor='#21262d', edgecolor='#30363d', alpha=0.9)
        ax.text(0.97, 0.97, textstr, transform=ax.transAxes, fontsize=9,
                verticalalignment='top', horizontalalignment='right', bbox=props, color='#e6edf3')
    else:
        ax.text(0.5, 0.5, 'No completed trades', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='#8b949e')

    ax.set_title(f'{agent} Agent', fontsize=11)
    ax.set_xlabel('Trade Return (%)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'rl_trade_distributions.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_trade_distributions.png')

## 5  Plot: Equity Curves — All Strategies

> ### 📝 TODO 10 (MEDIUM) — Normalize and plot the equity curve
>
> For each strategy, convert its raw portfolio-value history `pv` into a
> cumulative-return-percentage series relative to `INITIAL_CAPITAL`:
> `(pv / INITIAL_CAPITAL - 1) * 100`. You need to slice `pv` to align with
> `dates_arr` first (`pv[1:n+1]`, using the `n` already computed above) since
> `pv` includes the starting capital as its first entry, one more point than
> there are trading dates. Then plot it: call `ax.plot(...)` the same way the
> other strategies in this loop are plotted (see the loop above this TODO in
> earlier notebook cells for the exact style if you're unsure) — color, line
> width (`lw`), and line style (`ls`) are already provided per strategy, and
> the label should include the strategy name and its final return, e.g.
> `f'{label}  ({final_ret:+.1f}%)'`.

In [ ]:
# ─── EQUITY CURVES ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')

strategy_items = [
    ('DQN (RL)',      dqn_res, COLORS['DQN'],     2.5, '-'),
    ('PPO (RL)',      ppo_res, COLORS['PPO'],     2.5, '-'),
    ('S1: Regime',   s1_res,  COLORS['S1'],      1.5, '--'),
    ('S2: Momentum', s2_res,  COLORS['S2'],      1.5, '--'),
    ('S3: MeanRev',  s3_res,  COLORS['S3'],      1.5, '--'),
    ('S4: VolBreak', s4_res,  COLORS['S4'],      1.5, '--'),
    ('Buy & Hold',   bh_res,  COLORS['Buy&Hold'],1.0, ':'),
]

dates_arr = pd.to_datetime(test_df['date'].values)

for label, res, color, lw, ls in strategy_items:
    pv = res['portfolio_values']
    n = min(len(pv) - 1, len(dates_arr))
    # TODO 10 (MEDIUM): compute `normalized`, the cumulative % return series
    # for this strategy, then plot it. See the markdown cell above.
    raise NotImplementedError("TODO 10: normalize and plot the equity curve")

ax.axhline(0, color='white', linewidth=0.5, alpha=0.4)

# Shade RL outperformance vs Buy&Hold
dqn_pv = dqn_res['portfolio_values']
bh_pv  = bh_res['portfolio_values']
n = min(len(dqn_pv) - 1, len(bh_pv) - 1, len(dates_arr))
dqn_norm = (dqn_pv[1:n+1] / INITIAL_CAPITAL - 1) * 100
bh_norm  = (bh_pv[1:n+1]  / INITIAL_CAPITAL - 1) * 100
ax.fill_between(dates_arr[:n], dqn_norm, bh_norm,
                where=dqn_norm >= bh_norm,
                alpha=0.07, color=COLORS['DQN'], label='_DQN outperforms B&H')

ax.set_title('Cumulative Returns — All Strategies vs RL Agents (AAPL Test Period)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Cumulative Return (%)', fontsize=11)
ax.legend(loc='upper left', fontsize=10, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.0f%%'))

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'rl_equity_curves_comparison.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_equity_curves_comparison.png')

## 6  Plot: Drawdown Profiles

> ### 📝 TODO 11 (EASY) — Drawdown series
>
> From the portfolio-value array `pv`, compute:
> - `peak` — the running high-water mark (the max portfolio value seen *so
>   far* at each point in time). `np.maximum.accumulate` does exactly this.
> - `dd` — drawdown as a percentage, `(pv - peak) / peak * 100`. This will be
>   `0` at new highs and negative everywhere else.

In [ ]:
# ─── DRAWDOWN PROFILES ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(18, 6))
fig.patch.set_facecolor('#0d1117')

for label, res, color, lw, ls in strategy_items:
    pv   = res['portfolio_values']
    n    = min(len(pv), len(dates_arr) + 1)
    pv   = np.array(pv[:n])
    # TODO 11 (EASY): compute `peak` (running high-water mark) and `dd`
    # (drawdown %, <= 0) from `pv`, then keep the existing plotting line below.
    raise NotImplementedError("TODO 11: compute the drawdown series")
    ax.plot(dates_arr[:n-1], dd[1:n], color=color, linewidth=lw, linestyle=ls,
            alpha=0.9, label=f'{label}  (MDD: {dd.min():.1f}%)')

ax.fill_between(dates_arr, 0, -100, alpha=0.02, color='red')
ax.axhline(0, color='white', linewidth=0.5, alpha=0.4)

ax.set_title('Drawdown Profiles — All Strategies (AAPL Test Period)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.legend(loc='lower left', fontsize=9, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'rl_drawdown_comparison.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_drawdown_comparison.png')

## 7  Plot: Risk-Return Scatter

> ### 📝 TODO 12 (EASY) — Annualized volatility
>
> Compute `vol`, the annualized volatility of a daily-returns array `dr`, as
> a percentage: standard deviation of daily returns, scaled up by
> `sqrt(252)` (252 trading days/year), then converted to a percentage
> (`* 100`). If `dr` is empty, `vol` should be `0`.

In [ ]:
# ─── RISK-RETURN SCATTER ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
fig.patch.set_facecolor('#0d1117')

color_list = [COLORS['DQN'], COLORS['PPO'], COLORS['S1'], COLORS['S2'],
              COLORS['S3'], COLORS['S4'], COLORS['Buy&Hold']]

for (label, res, color, _, _), c in zip(strategy_items, color_list):
    dr  = res['daily_returns']
    # TODO 12 (EASY): compute `vol`, the annualized volatility (%) of `dr`.
    # Guard against the empty-array case (len(dr) == 0).
    raise NotImplementedError("TODO 12: compute annualized volatility")
    ret = res['ann_return']
    sh  = res['sharpe']

    is_rl = label.startswith('DQN') or label.startswith('PPO')
    marker = '*' if is_rl else 'o'
    size   = 350 if is_rl else 180

    ax.scatter(vol, ret, color=color, s=size, marker=marker,
               edgecolors='white', linewidths=0.8, zorder=5,
               label=f'{label}  (SR: {sh:.2f})', alpha=0.95)
    ax.annotate(label, (vol, ret), textcoords='offset points',
                xytext=(8, 3), fontsize=9, color=color)

# Efficient frontier reference line
ax.axhline(0, color='white', linewidth=0.5, alpha=0.3, linestyle='--')
ax.axvline(0, color='white', linewidth=0.5, alpha=0.3, linestyle='--')

ax.set_title('Risk-Return Tradeoff — All Strategies (AAPL Test Period)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Annualized Volatility (%)', fontsize=11)
ax.set_ylabel('Annualized Return (%)', fontsize=11)
ax.legend(fontsize=8, loc='upper left')

# Note: ★ marks RL agents
ax.text(0.98, 0.02, '★ = RL Agent  ● = Rule-Based',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=9, color='#8b949e')

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'rl_risk_return_scatter.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_risk_return_scatter.png')

## 8  Plot: Monthly Return Heatmaps (DQN & PPO)

In [ ]:
# ─── MONTHLY RETURN HEATMAPS ─────────────────────────────────────────────────
def monthly_heatmap(res, label, ax):
    pv   = res['portfolio_values']
    dates_arr = pd.to_datetime(test_df['date'].values)
    n = min(len(pv) - 1, len(dates_arr))

    dr_series = pd.Series(
        np.diff(pv[:n+1]) / pv[:n],
        index=dates_arr[:n]
    )

    # Aggregate to monthly
    monthly = dr_series.resample('ME').apply(lambda x: (1 + x).prod() - 1) * 100
    pivot   = monthly.to_frame('ret')
    pivot['Year']  = pivot.index.year
    pivot['Month'] = pivot.index.month
    heat = pivot.pivot(index='Year', columns='Month', values='ret')
    heat.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'][
        :heat.shape[1]]

    cmap = LinearSegmentedColormap.from_list(
        'rg', ['#ef4444', '#1f2937', '#22c55e'], N=200
    )
    sns.heatmap(heat, ax=ax, cmap=cmap, center=0,
                annot=True, fmt='.1f', annot_kws={'size': 8},
                linewidths=0.5, linecolor='#30363d',
                cbar_kws={'label': 'Monthly Return (%)', 'shrink': 0.8})
    ax.set_title(f'{label} — Monthly Returns (%)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Year')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)


fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.patch.set_facecolor('#0d1117')
monthly_heatmap(dqn_res, 'DQN Agent', axes[0])
monthly_heatmap(ppo_res, 'PPO Agent', axes[1])

plt.tight_layout(pad=2.0)
plt.savefig(os.path.join(REPORTS_DIR, 'rl_monthly_heatmaps.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_monthly_heatmaps.png')

## 9  Plot: Sharpe & Max Drawdown Comparison (Bar)

In [ ]:
# ─── BAR CHART: SHARPE & DRAWDOWN ────────────────────────────────────────────
labels_bar = ['DQN\n(RL)', 'PPO\n(RL)', 'S1\nRegime', 'S2\nMomentum',
               'S3\nMeanRev', 'S4\nVolBreak', 'Buy &\nHold']
sharpes = [r['sharpe']      for r in all_results]
mdds    = [abs(r['max_drawdown']) for r in all_results]
returns = [r['total_return'] for r in all_results]
bar_cols = [COLORS['DQN'], COLORS['PPO'], COLORS['S1'], COLORS['S2'],
            COLORS['S3'], COLORS['S4'], COLORS['Buy&Hold']]

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('Strategy Comparison — Key Metrics (AAPL Test Period)', fontsize=13, fontweight='bold', color='#e6edf3')

# Sharpe Ratio
bars = axes[0].bar(labels_bar, sharpes, color=bar_cols, alpha=0.85, edgecolor='#30363d', linewidth=0.8)
axes[0].set_title('Sharpe Ratio', fontsize=11)
axes[0].set_ylabel('Sharpe Ratio')
axes[0].axhline(0, color='white', linewidth=0.5, alpha=0.5)
for bar, val in zip(bars, sharpes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=8, color='#e6edf3')

# Max Drawdown
bars2 = axes[1].bar(labels_bar, mdds, color=bar_cols, alpha=0.85, edgecolor='#30363d', linewidth=0.8)
axes[1].set_title('Max Drawdown (%)', fontsize=11)
axes[1].set_ylabel('Max Drawdown (%)')
for bar, val in zip(bars2, mdds):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=8, color='#e6edf3')

# Total Return
bars3 = axes[2].bar(labels_bar, returns, color=bar_cols, alpha=0.85, edgecolor='#30363d', linewidth=0.8)
axes[2].set_title('Total Return (%)', fontsize=11)
axes[2].set_ylabel('Total Return (%)')
axes[2].axhline(0, color='white', linewidth=0.5, alpha=0.5)
for bar, val in zip(bars3, returns):
    ypos = bar.get_height() + (1 if val >= 0 else -3)
    axes[2].text(bar.get_x() + bar.get_width()/2, ypos,
                 f'{val:+.1f}%', ha='center', va='bottom', fontsize=8, color='#e6edf3')

for ax in axes:
    ax.tick_params(axis='x', labelsize=8)

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'rl_metrics_barchart.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_metrics_barchart.png')

## 10  Plot: DQN Training Curves (from saved results)

In [ ]:
# ─── RL TRAINING LEARNING CURVES ─────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')
gs = gridspec.GridSpec(2, 3, hspace=0.35, wspace=0.3)
fig.suptitle('RL Training Learning Curves (AAPL)', fontsize=13, fontweight='bold', color='#e6edf3')

dqn_ret = rl_results['dqn_episode_returns']
dqn_loss= rl_results['dqn_episode_losses']
dqn_eps = rl_results['dqn_epsilon_history']
ppo_loss= rl_results['ppo_epoch_losses']
ppo_ret = rl_results['ppo_epoch_returns']

# DQN Episode Returns
ax = fig.add_subplot(gs[0, 0])
ax.plot(dqn_ret, alpha=0.4, color=COLORS['DQN'], linewidth=0.7)
ax.plot(pd.Series(dqn_ret).rolling(20).mean(), color=COLORS['DQN'], linewidth=2, label='20-ep MA')
ax.axhline(0, color='white', linewidth=0.5, alpha=0.5, linestyle='--')
ax.set_title('DQN Episode Returns (%)')
ax.set_xlabel('Episode'); ax.legend(fontsize=8)

# DQN Loss
ax = fig.add_subplot(gs[0, 1])
ax.plot(dqn_loss, alpha=0.4, color='tomato', linewidth=0.7)
ax.plot(pd.Series(dqn_loss).rolling(20).mean(), color='tomato', linewidth=2, label='20-ep MA')
ax.set_title('DQN Training Loss')
ax.set_xlabel('Episode'); ax.legend(fontsize=8)

# DQN Epsilon
ax = fig.add_subplot(gs[0, 2])
ax.plot(dqn_eps, color='#a78bfa', linewidth=1.5)
ax.set_title('DQN Epsilon Decay')
ax.set_xlabel('Episode')
ax.set_ylabel('ε')
ax.fill_between(range(len(dqn_eps)), dqn_eps, 0, alpha=0.15, color='#a78bfa')

# PPO Loss
ax = fig.add_subplot(gs[1, 0])
ax.plot(ppo_loss, alpha=0.4, color='tomato', linewidth=0.7)
ax.plot(pd.Series(ppo_loss).rolling(20).mean(), color='tomato', linewidth=2, label='20-ep MA')
ax.set_title('PPO Training Loss')
ax.set_xlabel('Epoch'); ax.legend(fontsize=8)

# PPO Val Returns
ax = fig.add_subplot(gs[1, 1])
ep_labels = list(range(0, len(ppo_ret) * 10, 10))
ax.plot(ep_labels, ppo_ret, color=COLORS['PPO'], linewidth=1.5, marker='o', markersize=4)
ax.axhline(0, color='white', linewidth=0.5, alpha=0.5, linestyle='--')
ax.set_title('PPO Validation Returns (%)')
ax.set_xlabel('Epoch')
ax.fill_between(ep_labels, ppo_ret, 0, alpha=0.1, color=COLORS['PPO'])

# Combined: Return convergence
ax = fig.add_subplot(gs[1, 2])
dqn_smooth = pd.Series(dqn_ret).rolling(30).mean().dropna().values
ep_x = range(30, len(dqn_ret))
ax.plot(ep_x, dqn_smooth, color=COLORS['DQN'], linewidth=2, label='DQN (30-ep MA)')
ppo_x = ep_labels
ax.plot(ppo_x, ppo_ret, color=COLORS['PPO'], linewidth=2, marker='s', markersize=4, label='PPO Val')
ax.axhline(0, color='white', linewidth=0.5, alpha=0.5, linestyle='--')
ax.set_title('DQN vs PPO Return Convergence')
ax.set_xlabel('Episode / Epoch')
ax.legend(fontsize=8)

plt.savefig(os.path.join(REPORTS_DIR, 'rl_training_overview.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_training_overview.png')

## 11  Final Comparison Table (Publication-Quality)

In [ ]:
# ─── FINAL SUMMARY TABLE (as figure) ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')
ax.axis('off')

# Build table data
col_labels = ['Strategy', 'Total Return', 'Ann. Return', 'Sharpe', 'Max DD', 'Win Rate', '# Trades']
cell_data  = []
row_colors = []

color_map_rows = [
    '#1a2a3a', '#1a2a3a',   # DQN, PPO (RL)
    '#1a1f2e', '#1a1f2e', '#1a1f2e', '#1a1f2e',  # S1–S4
    '#111827',              # B&H
]

for label, res in zip(labels, all_results):
    cell_data.append([
        label,
        f"{res['total_return']:+.2f}%",
        f"{res['ann_return']:+.2f}%",
        f"{res['sharpe']:.3f}",
        f"{res['max_drawdown']:.2f}%",
        f"{res['win_rate']:.1f}%",
        str(res['n_trades'])
    ])

# Sort by sharpe
sharpe_order = np.argsort([-r['sharpe'] for r in all_results])
cell_data_sorted = [cell_data[i] for i in sharpe_order]
color_map_sorted = [color_map_rows[i] for i in sharpe_order]

table = ax.table(
    cellText=cell_data_sorted,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)
table.auto_set_font_size(False)
table.set_fontsize(10)

# Style header
for j, _ in enumerate(col_labels):
    cell = table[0, j]
    cell.set_facecolor('#21262d')
    cell.set_text_props(color='#e6edf3', fontweight='bold')
    cell.set_edgecolor('#30363d')

# Style rows
strategy_color_map = {
    'DQN (RL)': COLORS['DQN'],
    'PPO (RL)': COLORS['PPO'],
}

for i, row_data in enumerate(cell_data_sorted):
    for j in range(len(col_labels)):
        cell = table[i+1, j]
        cell.set_facecolor(color_map_sorted[i])
        cell.set_edgecolor('#30363d')
        tc = '#e6edf3'
        # Highlight strategy name
        if j == 0 and row_data[0] in strategy_color_map:
            tc = strategy_color_map[row_data[0]]
        # Color returns
        if j in (1, 2):  # return columns
            val_str = row_data[j].replace('%','').replace('+','')
            try:
                val = float(val_str)
                tc = COLORS['profit'] if val > 0 else COLORS['loss']
            except: pass
        cell.set_text_props(color=tc)

ax.set_title('Final Strategy Comparison — AAPL Test Period (Sorted by Sharpe Ratio)',
             fontsize=12, fontweight='bold', color='#e6edf3', pad=20)

plt.savefig(os.path.join(REPORTS_DIR, 'rl_final_comparison_table.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_final_comparison_table.png')

## 12  Rolling Sharpe Comparison

> ### 📝 TODO 13 (MEDIUM) — Rolling Sharpe ratio
>
> Compute a rolling Sharpe ratio over `dr_s` (daily returns) using a
> `WINDOW`-day window: `.rolling(WINDOW).apply(...)`. Inside the rolling
> window function, the Sharpe ratio of a return series `x` is
> `(x.mean() / x.std()) * sqrt(252)` (252 trading days/year annualizes it).
> Add a small epsilon (e.g. `1e-8`) to the denominator to avoid a
> divide-by-zero on windows with zero variance.

In [ ]:
# ─── ROLLING SHARPE RATIO ────────────────────────────────────────────────────
WINDOW = 60  # 60-day rolling window

fig, ax = plt.subplots(figsize=(18, 6))
fig.patch.set_facecolor('#0d1117')

for label, res, color, lw, ls in strategy_items:
    dr  = res['daily_returns']
    n   = min(len(dr), len(dates_arr) - 1)
    dr_s = pd.Series(dr[:n])
    # TODO 13 (MEDIUM): compute `rolling_sh`, the WINDOW-day rolling Sharpe
    # ratio of `dr_s`. See the markdown cell above.
    raise NotImplementedError("TODO 13: compute the rolling Sharpe ratio")
    ax.plot(dates_arr[WINDOW:n], rolling_sh.values[WINDOW:], color=color,
            linewidth=lw, linestyle=ls, alpha=0.85, label=label)

ax.axhline(0, color='white', linewidth=0.5, alpha=0.4)
ax.axhline(1.0, color='#22c55e', linewidth=0.5, alpha=0.3, linestyle=':')
ax.text(dates_arr[WINDOW], 1.05, 'Sharpe = 1.0', color='#22c55e', fontsize=8, alpha=0.7)

ax.set_title(f'{WINDOW}-Day Rolling Sharpe Ratio — All Strategies (AAPL Test Period)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel(f'{WINDOW}-Day Rolling Sharpe')
ax.legend(fontsize=9, loc='upper left', ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(REPORTS_DIR, 'rl_rolling_sharpe.png'), dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Saved: rl_rolling_sharpe.png')

## 13  Final Summary

In [ ]:
# ─── FINAL SUMMARY ───────────────────────────────────────────────────────────
print('=' * 70)
print('  FINAL RL TRADING AGENT RESULTS — AAPL')
print('=' * 70)
print(f'  Test period: {pd.to_datetime(test_dates[0]).date()} → {pd.to_datetime(test_dates[-1]).date()}')
print(f'  Initial Capital: ${INITIAL_CAPITAL:,.0f}')
print()
print(summary_df.sort_values('Sharpe Ratio', ascending=False).to_string(index=False))
print()

# Outperformance check
bh_sharpe = bh_res['sharpe']
bh_ret    = bh_res['total_return']

for label, res in [('DQN', dqn_res), ('PPO', ppo_res)]:
    beats_sharpe = res['sharpe'] > bh_sharpe
    beats_return = res['total_return'] > bh_ret
    print(f'  {label}: Beats B&H Sharpe? {"✅ YES" if beats_sharpe else "❌ NO"} '
          f'({res["sharpe"]:.3f} vs {bh_sharpe:.3f}) | '
          f'Beats B&H Return? {"✅ YES" if beats_return else "❌ NO"} '
          f'({res["total_return"]:+.2f}% vs {bh_ret:+.2f}%)')

print()
print('Reports saved:')
reports = [
    'dqn_trades_analysis.png', 'ppo_trades_analysis.png',
    'rl_trade_distributions.png', 'rl_equity_curves_comparison.png',
    'rl_drawdown_comparison.png', 'rl_risk_return_scatter.png',
    'rl_monthly_heatmaps.png', 'rl_metrics_barchart.png',
    'rl_training_overview.png', 'rl_final_comparison_table.png',
    'rl_rolling_sharpe.png', 'full_strategy_comparison.csv',
    'rl_agent_summary.csv'
]
for r in reports:
    path = os.path.join(REPORTS_DIR, r)
    exists = '✅' if os.path.exists(path) else '⏳'
    print(f'  {exists} {r}')